In [ ]:
# ========== 第 1 周练习 2：抓网页 → 本地 Ollama 生成 Markdown 摘要 ==========
# 本单元格一次做完四件事：
# 1) 确保本机有 Ollama 模型 llama3.2:1b（没有就 pull）
# 2) 用 scraper.fetch_website_contents 抓取目标网页正文
# 3) 经 OpenAI 兼容接口把正文发给本地 LLM
# 4) 打印 Markdown 格式的页面摘要
# 和本课关系：Day2「网页抓取 + 本地模型总结」；base_url 指向本机而不是云端

# 确保 Ollama 已安装、正在运行，并监听 localhost:11434
# 若本地还没有 llama3.2:1b，这行 shell 魔法会拉取模型（需网络与磁盘空间）
!ollama pull llama3.2:1b
# 从 openai 导入 OpenAI 客户端：下面会把它的 base_url 指到本地 Ollama
from openai import OpenAI
# 从同目录 scraper 模块导入抓取函数：把网页变成纯文本字符串
from scraper import fetch_website_contents

# ---------- 配置：本地 Ollama 的 OpenAI 兼容地址 + 要总结的网页 ----------

# Ollama 提供的 OpenAI 兼容 API 根路径（注意末尾 /v1/）
ollama_url = "http://localhost:11434/v1/"
# 目标网页 URL：关于作者与 Nebula 的介绍页（字符串保持原样）
webpage = "https://edwarddonner.com/about-me-and-about-nebula/"

# 抓取网页正文，得到给模型阅读的原始字符串
webpage_contents = fetch_website_contents(webpage)

# ---------- 组装 chat messages：system 定格式，user 塞网页正文 ----------
# - system：要求用 Markdown 给出优秀摘要（prompt 字符串不翻译）
# - user：把网页内容嵌进提示
prompts = [
    {"role":"system","content":"You provide and excellent summary of the webpage contents in a Markdown format"},
    {"role":"user","content":f"Here are the webpage contents: {webpage_contents}"}
]

# 创建指向本地 Ollama 的 OpenAI 客户端（不是云端 OpenAI）
# api_key 会被 Ollama 忽略，但 SDK 要求非空，所以填占位 "ollama"
ollama = OpenAI(base_url = ollama_url, api_key = "ollama")

# 调用本地模型的 chat.completions；model 必须是本机已有的名字
response = ollama.chat.completions.create(messages=prompts, model="llama3.2:1b")

# 摘要在 choices[0].message.content；打印出来便于阅读 Markdown
print(response.choices[0].message.content)
